In [1]:
import numpy as np
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

# ==========================================
# SETUP: Load Documents & Initialize Embedder
# ==========================================
print("Loading data from GitHub...")
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
embed = Embedder()

# ==========================================
# Q1: Embedding a query
# ==========================================
q1_query = "How does approximate nearest neighbor search work?"
v = embed.encode(q1_query)
print(f"\n--- Q1 ---")
print(f"The first value (v[0]) is: {v[0]:.2f}")

# ==========================================
# Q2: Cosine similarity
# ==========================================
# Find the specific target document from the list
doc_q2 = next(d for d in documents if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md')
doc_v = embed.encode(doc_q2['content'])

# Because the vectors are normalized by the ONNX embedder, dot product equals cosine similarity
sim = v.dot(doc_v)
print(f"\n--- Q2 ---")
print(f"Cosine similarity: {sim:.2f}")

# ==========================================
# Q3: Chunking and search by hand
# ==========================================
print("\nChunking documents and creating embeddings matrix (this may take a moment)...")
chunks = chunk_documents(documents, size=2000, step=1000)
texts = [c['content'] for c in chunks]

# Embed in batches of 50 just like the lesson
X = []
for i in range(0, len(texts), 50):
    batch = texts[i:i+50]
    X.extend(embed.encode_batch(batch))
X = np.array(X) # Your Embedding Matrix

# Search by hand
scores = X.dot(v)
best_idx = np.argmax(scores)
print(f"\n--- Q3 ---")
print(f"Top file (by hand): {chunks[best_idx]['filename']}")

# ==========================================
# Q4: Vector search with minsearch
# ==========================================
q4_query = "What metric do we use to evaluate a search engine?"
q4_v = embed.encode(q4_query)

# Use the VectorSearch class from minsearch
vec_index = VectorSearch(keyword_fields=["filename"])
vec_index.fit(X, chunks)
results_q4 = vec_index.search(q4_v, num_results=1)

print(f"\n--- Q4 ---")
print(f"Top file (minsearch): {results_q4[0]['filename']}")

# ==========================================
# Q5: Text search vs vector search
# ==========================================
q5_query = "How do I store vectors in PostgreSQL?"
q5_v = embed.encode(q5_query)

# Create a standard Keyword Index for comparison
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

text_results = text_index.search(q5_query, num_results=5)
vec_results = vec_index.search(q5_v, num_results=5)

# Extract just the filenames to compare them
text_filenames = [r['filename'] for r in text_results]
vec_filenames = [r['filename'] for r in vec_results]

# Find the file that exists in the Vector list, but NOT the Text list
q5_ans = set(vec_filenames) - set(text_filenames)

print(f"\n--- Q5 ---")
print(f"File found by Vector but missed by Text: {list(q5_ans)}")

# ==========================================
# Q6: Hybrid search (RRF)
# ==========================================
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q6_query = "How do I give the model access to tools?"
q6_v = embed.encode(q6_query)

q6_text_results = text_index.search(q6_query, num_results=5)
q6_vec_results = vec_index.search(q6_v, num_results=5)

hybrid_results = rrf([q6_vec_results, q6_text_results])

print(f"\n--- Q6 ---")
print(f"Top file after RRF Hybrid Search: {hybrid_results[0]['filename']}")

2026-06-29 21:02:45.468135386 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Loading data from GitHub...

--- Q1 ---
The first value (v[0]) is: -0.02

--- Q2 ---
Cosine similarity: 0.36

Chunking documents and creating embeddings matrix (this may take a moment)...

--- Q3 ---
Top file (by hand): 02-vector-search/lessons/07-sqlitesearch-vector.md

--- Q4 ---
Top file (minsearch): 04-evaluation/lessons/05-search-metrics.md

--- Q5 ---
File found by Vector but missed by Text: ['02-vector-search/lessons/08-pgvector.md']

--- Q6 ---
Top file after RRF Hybrid Search: 01-agentic-rag/lessons/13-function-calling.md
